In [1]:
import numpy as np
import pandas as pd
import warnings
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import torch

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')
%env JOBLIB_TEMP_FOLDER=/tmp

env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
# folder_path = 'dataset/'
from pathlib import Path
import pandas as pd

# Detect environment
try:
    import google.colab

    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")

Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [4]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_random_forest(X_train, X_valid, y_train, y_valid, name="Random Forest"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [5]:
# LightGBM
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_lightgbm(X_train, X_valid, y_train, y_valid, name="LightGBM"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [6]:
# XGBoost
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def run_xgboost(X_train, X_valid, y_train, y_valid, name="XGBoost"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
    }

In [7]:
# DATA SPLIT / EXPERIMENT DATASETS
from itertools import combinations

train = train.sort_values("TransactionDT").reset_index(drop=True)

y = train["isFraud"]

split_idx = int(len(train) * 0.8)

y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]


# Baseline
# Original features, excluding engineered UID / UID2

baseline_cols = [
    col
    for col in train.columns
    if col not in ["isFraud", "TransactionID", "uid", "uid2"]
]

X_baseline = train[baseline_cols]

X_train_baseline = X_baseline.iloc[:split_idx]
X_valid_baseline = X_baseline.iloc[split_idx:]


# Feature Engineering
# Original + temporal + UID + UID2

feature_cols = [col for col in train.columns if col not in ["isFraud", "TransactionID"]]

X_features = train[feature_cols]

X_train_features = X_features.iloc[:split_idx]
X_valid_features = X_features.iloc[split_idx:]


# Feature-group ablation

prefix_groups = {
    "C": ("C",),
    "D": ("D",),
    "M": ("M",),
    "id": ("id_",),
    "V": ("V",),
}

group_names = list(prefix_groups.keys())

ablation_datasets = {}

for r in range(1, len(group_names) + 1):

    for combination in combinations(group_names, r):

        prefixes = tuple(
            prefix for group in combination for prefix in prefix_groups[group]
        )

        selected_cols = [col for col in baseline_cols if not col.startswith(prefixes)]

        X = train[selected_cols]

        name = "Remove " + " + ".join(combination)

        ablation_datasets[name] = (X.iloc[:split_idx], X.iloc[split_idx:])

print("Baseline features :", len(baseline_cols))
print("Feature engineering:", len(feature_cols))
print("Ablation experiments:", len(ablation_datasets))

MemoryError: Unable to allocate 865. MiB for an array with shape (384, 590540) and data type float32

In [ ]:
# ALL ML EXPERIMENTS
rf_results = []
lgb_results = []
xgb_results = []

In [ ]:
def save_results():

    def prepare(results):
        df = pd.DataFrame(results).copy()

        # Convert model objects to model names
        if "model" in df.columns:
            df["model"] = df["model"].apply(lambda x: type(x).__name__)

        # Convert confusion matrix to TN/FP/FN/TP
        if "Confusion Matrix" in df.columns:
            cm = df.pop("Confusion Matrix").tolist()

            df["TN"] = [x[0][0] for x in cm]
            df["FP"] = [x[0][1] for x in cm]
            df["FN"] = [x[1][0] for x in cm]
            df["TP"] = [x[1][1] for x in cm]

        return df

    prepare(rf_results).to_parquet(f"{SAVED_PATH}/rf_results.parquet", index=False)

    prepare(lgb_results).to_parquet(f"{SAVED_PATH}/lgb_results.parquet", index=False)

    prepare(xgb_results).to_parquet(f"{SAVED_PATH}/xgb_results.parquet", index=False)

In [ ]:
# 1. BASELINE - RF

rf_results.append(
    run_random_forest(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "RF - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING

rf_results.append(
    run_random_forest(
        X_train_features, X_valid_features, y_train, y_valid, "RF - Feature Engineering"
    )
)

save_results()

In [ ]:
# 1. BASELINE - LGBM
lgb_results.append(
    run_lightgbm(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "LightGBM - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
lgb_results.append(
    run_lightgbm(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "LightGBM - Feature Engineering",
    )
)
save_results()

In [ ]:
# 1. BASELINE - XGB
xgb_results.append(
    run_xgboost(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "XGBoost - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
xgb_results.append(
    run_xgboost(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "XGBoost - Feature Engineering",
    )
)
save_results()

In [ ]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():

    rf_results.append(
        run_random_forest(X_train, X_valid, y_train, y_valid, f"RF - {name}")
    )
    save_results()

In [ ]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    lgb_results.append(
        run_lightgbm(X_train, X_valid, y_train, y_valid, f"LightGBM - {name}")
    )

    save_results()

In [ ]:

# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    xgb_results.append(
        run_xgboost(
            X_train, X_valid,
            y_train, y_valid,
            f"XGBoost - {name}"
        )
    )

    save_results()

In [ ]:
# RESULTS

metrics = [
    "Model",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
]


rf_results_df = pd.DataFrame(rf_results)
lgb_results_df = pd.DataFrame(lgb_results)
xgb_results_df = pd.DataFrame(xgb_results)

print("===== RANDOM FOREST =====")
display(
    rf_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

print("===== LIGHTGBM =====")
display(
    lgb_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

print("===== XGBOOST =====")
display(
    xgb_results_df[metrics]
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline


def get_sampling_variants(X_train, y_train):
    variants = {
        "Original": (X_train, y_train),
        "SMOTE": SMOTE(random_state=42).fit_resample(X_train, y_train),
        "Undersampling": RandomUnderSampler(random_state=42).fit_resample(
            X_train, y_train
        ),
        "SMOTE + Undersampling": Pipeline(
            [
                ("under", RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
                ("smote", SMOTE(sampling_strategy=0.5, random_state=42)),
            ]
        ).fit_resample(X_train, y_train),
    }

    return variants

In [ ]:
# ALL FEATURE CONFIGURATIONS
feature_datasets = {
    "Baseline": (X_train_baseline, X_valid_baseline),
    "Feature Engineering": (X_train_features, X_valid_features),
    **{
        name: (X_train, X_valid)
        for name, (X_train, X_valid) in ablation_datasets.items()
    },
}

In [ ]:
# RUN EXPERIMENTS


for feature_name, (X_train, X_valid) in feature_datasets.items():

    sampling_variants = get_sampling_variants(X_train, y_train)

    for sampling_name, (X_sampled, y_sampled) in sampling_variants.items():

        experiment_name = f"{feature_name} - {sampling_name}"

        print(f"\n===== {experiment_name} =====")
        print(
            f"Train samples: "
            f"{len(y_sampled):,} | "
            f"Fraud rate: {y_sampled.mean():.4f}"
        )

        # Random Forest
        rf_results.append(
            run_random_forest(
                X_sampled, X_valid, y_sampled, y_valid, f"RF - {experiment_name}"
            )
        )
        save_results()

        # LightGBM
        lgb_results.append(
            run_lightgbm(
                X_sampled, X_valid, y_sampled, y_valid, f"LightGBM - {experiment_name}"
            )
        )
        save_results()

        # XGBoost
        xgb_results.append(
            run_xgboost(
                X_sampled, X_valid, y_sampled, y_valid, f"XGBoost - {experiment_name}"
            )
        )
        save_results()